In [2]:
# from google.colab import drive  # Colab-only: commented out for local run; RESTORE before sharing with groupmates on Colab
import numpy as np
import pandas as pd
import os
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import chain
from itertools import combinations

In [3]:
# ============================================================
# COLAB VERSION (commented out) - RESTORE this block before sharing on Colab
# ============================================================
# # 1) Mount
# drive.mount("/content/drive", force_remount=True)
#
# # 2) Paths
# BASE = Path("/content/drive/MyDrive/bt4222/JDsearch")
# PRODUCT_TXT = BASE / "product_meta_data.txt"
# USER_TXT    = BASE / "user_behavior_data.txt"
# CHUNKS_DIR  = BASE / "product_chunks"
# CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LOCAL VERSION (active) - reads the extracted JDsearch.tar contents
# from ./JDsearch/ next to this notebook
# ============================================================
BASE = Path("JDsearch")
PRODUCT_TXT = BASE / "product_meta_data.txt"
USER_TXT    = BASE / "user_behavior_data.txt"
CHUNKS_DIR  = BASE / "product_chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# 3) Load small user data
user_df = pd.read_csv(USER_TXT, sep="\t", dtype=str, low_memory=False)
print("user_behaviour shape:", user_df.shape)

user_behaviour shape: (173831, 7)


In [4]:
user_df[user_df['candidate_label_list'].apply(len) == 0]

,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list


In [5]:
# --- HELPER FUNCTIONS ---
def safe_split(val, sep='_'):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return str(val).split(sep)

def safe_split_numeric(val, sep='_'):
    lst = safe_split(val, sep)
    return [float(x) for x in lst]

# 1. PARSING THE DATAFRAME
user_df['candidate_wid_list_parsed'] = user_df['candidate_wid_list'].apply(safe_split_numeric)
user_df['candidate_label_list_parsed'] = user_df['candidate_label_list'].apply(safe_split_numeric)
user_df['history_qry_list_parsed'] = user_df['history_qry_list'].apply(safe_split)
user_df['history_wid_list_parsed'] = user_df['history_wid_list'].apply(safe_split)
user_df['history_type_list_parsed'] = user_df['history_type_list'].apply(safe_split)
user_df['history_time_list_parsed'] = user_df['history_time_list'].apply(safe_split_numeric)

In [6]:
user_df.head()

,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list,candidate_wid_list_parsed,candidate_label_list_parsed,history_qry_list_parsed,history_wid_list_parsed,history_type_list_parsed,history_time_list_parsed
0,63995226602574196100244625575117,36098226_17729172_34921664_68317377_693764_360...,1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1_-1_68784030_36547097602574193663289_95...,9227588_52788392_4585436_87422901_57364733_210...,ORD_ORD_CART_CART_CART_CART_CART_CART_CART_CAR...,0_0_22967659_82448_432_10315_51_188_51_254173_...,"[36098226.0, 17729172.0, 34921664.0, 68317377....","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-1, -1, -1, 68784030, 365470976025741936632...","[9227588, 52788392, 4585436, 87422901, 5736473...","[ORD, ORD, CART, CART, CART, CART, CART, CART,...","[0.0, 0.0, 22967659.0, 82448.0, 432.0, 10315.0..."
1,1254963427438884,30857980_44577431_1485384_63372823_19754701_29...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,9755613572694219_46916741_-1_-1_-1_-1_-1_4927...,18256630_46206270_65230659_86878921_32743602_5...,ORD_ORD_ORD_CART_CART_CART_CART_ORD_CART_CART_...,0_4813478_1115463_14061983_920_467229_304_1197...,"[30857980.0, 44577431.0, 1485384.0, 63372823.0...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[9755613572694219, 46916741, -1, -1, -1, -1, ...","[18256630, 46206270, 65230659, 86878921, 32743...","[ORD, ORD, ORD, CART, CART, CART, CART, ORD, C...","[0.0, 4813478.0, 1115463.0, 14061983.0, 920.0,..."
2,24374376067768939206305,42155506_90092899_4336287_74203839_39478459_37...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1,8133925_59478082,CLICK_CLICK,0_27_49660,"[42155506.0, 90092899.0, 4336287.0, 74203839.0...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-1, -1]","[8133925, 59478082]","[CLICK, CLICK]","[0.0, 27.0, 49660.0]"
3,29463948632095349356845164880503931105432...,97234676_89827009_19973201_83859812_22172053_3...,1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,3426234151020854598336931355834712920960_3...,49854616_56947170_16552251_91485122_25087051_7...,CLICK_CLICK_CLICK_CLICK_CLICK_CLICK_CLICK_CLIC...,0_12_4_6_4_14_6_6_4_611_58095_1524_481_20_363_...,"[97234676.0, 89827009.0, 19973201.0, 83859812....","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[3426234151020854598336931355834712920960,...","[49854616, 56947170, 16552251, 91485122, 25087...","[CLICK, CLICK, CLICK, CLICK, CLICK, CLICK, CLI...","[0.0, 12.0, 4.0, 6.0, 4.0, 14.0, 6.0, 6.0, 4.0..."
4,2793914964365783663289,19711065_83045333_8307571_20337381_11912759_53...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,25499566_24358647_606776896025741971768678_3...,68701_66196939_45800470_18419519_10090448_8073...,ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_ORD_OR...,0_0_2015453_958482_0_79443_854291_13678_119386...,"[19711065.0, 83045333.0, 8307571.0, 20337381.0...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[25499566, 24358647, 60677689602574197176867...","[68701, 66196939, 45800470, 18419519, 10090448...","[ORD, ORD, ORD, ORD, ORD, ORD, ORD, ORD, ORD, ...","[0.0, 0.0, 2015453.0, 958482.0, 0.0, 79443.0, ..."


### Numeric Features

In [7]:
# ============================================================
# FEATURE 1: candidate_count
# Number of candidates returned for the current query
# ============================================================

user_df["candidate_count"] = user_df["candidate_wid_list_parsed"].apply(
    lambda x: len(x) if isinstance(x, list) else np.nan
)

print("candidate_count")
print(f"Missing/unparseable lists: {user_df['candidate_count'].isna().sum():,}")
print(f"Zero candidate lists:      {(user_df['candidate_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['candidate_count'] < 0).sum():,}")

# ============================================================
# FEATURE 2: history_unique_query_count
# Number of distinct valid historical queries
# Excludes -1 (non-search interactions)
# ============================================================

def count_unique_queries(x):
    if not isinstance(x, list):
        return np.nan

    valid_queries = [q for q in x if q != -1]

    return len(set(valid_queries))


user_df["history_unique_query_count"] = (
    user_df["history_qry_list_parsed"].apply(count_unique_queries)
)

print("history_unique_query_count")
print(f"Missing/unparseable lists: {user_df['history_unique_query_count'].isna().sum():,}")
print(f"Zero unique queries:       {(user_df['history_unique_query_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['history_unique_query_count'] < 0).sum():,}")

# ============================================================
# FEATURE 3: history_unique_wid_count
# Number of distinct products interacted with historically
# ============================================================

user_df["history_unique_wid_count"] = user_df["history_wid_list_parsed"].apply(
    lambda x: len(set(x)) if isinstance(x, list) else np.nan
)

print("history_unique_wid_count")
print(f"Missing/unparseable lists: {user_df['history_unique_wid_count'].isna().sum():,}")
print(f"Zero unique WIDs:          {(user_df['history_unique_wid_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['history_unique_wid_count'] < 0).sum():,}")

candidate_count
Missing/unparseable lists: 0
Zero candidate lists:      0
Negative values:           0
history_unique_query_count
Missing/unparseable lists: 0
Zero unique queries:       0
Negative values:           0
history_unique_wid_count
Missing/unparseable lists: 0
Zero unique WIDs:          0
Negative values:           0


In [8]:
# ============================================================
# FEATURES 4-6: Candidate interaction rates
# IMPORTANT: Replace with actual JDSearch label mapping
# ============================================================

CLICK_LABEL = 1.0
CART_LABEL = 2.0
PURCHASE_LABEL = 3.0

def candidate_rate(labels, target_label):
    """
    Proportion of candidates receiving target_label.

    Returns NaN when:
    - parsed value is not a list
    - candidate label list is empty
    """
    if not isinstance(labels, list) or len(labels) == 0:
        return np.nan

    return sum(label == target_label for label in labels) / len(labels)


# Check denominator problems before calculating rates
invalid_label_lists = ~user_df["candidate_label_list_parsed"].apply(
    lambda x: isinstance(x, list)
)

empty_label_lists = user_df["candidate_label_list_parsed"].apply(
    lambda x: isinstance(x, list) and len(x) == 0
)

print("Candidate rate denominator checks")
print(f"Unparseable/non-list values: {invalid_label_lists.sum():,}")
print(f"Empty lists (zero denominator): {empty_label_lists.sum():,}")
print("These observations will receive NaN for candidate rates.")


# Calculate rates
user_df["candidate_click_rate"] = (
    user_df["candidate_label_list_parsed"]
    .apply(lambda x: candidate_rate(x, CLICK_LABEL))
)

user_df["candidate_cart_rate"] = (
    user_df["candidate_label_list_parsed"]
    .apply(lambda x: candidate_rate(x, CART_LABEL))
)

user_df["candidate_purchase_rate"] = (
    user_df["candidate_label_list_parsed"]
    .apply(lambda x: candidate_rate(x, PURCHASE_LABEL))
)


# Validate resulting rates
for col in [
    "candidate_click_rate",
    "candidate_cart_rate",
    "candidate_purchase_rate"
]:
    print(f"\n{col}")
    print(f"Missing:       {user_df[col].isna().sum():,}")
    print(f"Below 0:       {(user_df[col] < 0).sum():,}")
    print(f"Above 1:       {(user_df[col] > 1).sum():,}")

Candidate rate denominator checks
Unparseable/non-list values: 0
Empty lists (zero denominator): 0
These observations will receive NaN for candidate rates.

candidate_click_rate
Missing:       0
Below 0:       0
Above 1:       0

candidate_cart_rate
Missing:       0
Below 0:       0
Above 1:       0

candidate_purchase_rate
Missing:       0
Below 0:       0
Above 1:       0


In [9]:
# ============================================================
# FEATURES 7-10: Historical action counts
# CLICK, CART, ORD, FOLLOW
# ============================================================

ACTION_MAP = {
    "history_click_count": "CLICK",
    "history_cart_count": "CART",
    "history_order_count": "ORD",
    "history_follow_count": "FOLLOW"
}

for col, action in ACTION_MAP.items():

    user_df[col] = user_df["history_type_list_parsed"].apply(
        lambda x: x.count(action) if isinstance(x, list) else np.nan
    )

    print(f"\n{col}")
    print(f"Missing/unparseable lists: {user_df[col].isna().sum():,}")
    print(f"Zero counts:               {(user_df[col] == 0).sum():,}")
    print(f"Negative values:           {(user_df[col] < 0).sum():,}")


history_click_count
Missing/unparseable lists: 0
Zero counts:               28,184
Negative values:           0

history_cart_count
Missing/unparseable lists: 0
Zero counts:               7,449
Negative values:           0

history_order_count
Missing/unparseable lists: 0
Zero counts:               14,686
Negative values:           0

history_follow_count
Missing/unparseable lists: 0
Zero counts:               173,831
Negative values:           0


In [10]:
# ============================================================
# FEATURE 11: history_length
# Number of historically interacted products
# ============================================================

user_df["history_length"] = user_df["history_wid_list_parsed"].apply(
    lambda x: len(x) if isinstance(x, list) else np.nan
)

print("history_length")
print(f"Missing/unparseable lists: {user_df['history_length'].isna().sum():,}")
print(f"Zero-length histories:     {(user_df['history_length'] == 0).sum():,}")
print(f"Negative values:           {(user_df['history_length'] < 0).sum():,}")

# ============================================================
# FEATURE 12: candidate_seen_history_count
# Number of current candidates previously interacted with
# ============================================================

def count_seen_candidates(row):
    candidates = row["candidate_wid_list_parsed"]
    history = row["history_wid_list_parsed"]

    if not isinstance(candidates, list) or not isinstance(history, list):
        return np.nan

    return len(set(candidates) & set(history))


user_df["candidate_seen_history_count"] = user_df.apply(
    count_seen_candidates,
    axis=1
)

print("candidate_seen_history_count")
print(f"Missing/unparseable lists: {user_df['candidate_seen_history_count'].isna().sum():,}")
print(f"Zero overlap:              {(user_df['candidate_seen_history_count'] == 0).sum():,}")
print(f"Negative values:           {(user_df['candidate_seen_history_count'] < 0).sum():,}")

# Logical consistency check
invalid_overlap = (
    user_df["candidate_seen_history_count"] > user_df["candidate_count"]
)

print(
    f"Count exceeding candidate_count: "
    f"{invalid_overlap.sum():,}"
)

history_length
Missing/unparseable lists: 0
Zero-length histories:     0
Negative values:           0
candidate_seen_history_count
Missing/unparseable lists: 0
Zero overlap:              173,831
Negative values:           0
Count exceeding candidate_count: 0


In [11]:
# ============================================================
# FEATURE 13: time_since_last_action
# ============================================================

def get_last_interval(time_list):
    if len(time_list) > 0:
        return float(time_list[-1])
    return np.nan

user_df["time_since_last_action"] = (
    user_df["history_time_list_parsed"].apply(get_last_interval)
)

# Data quality checks
print("time_since_last_action")
print(f"Missing values:   {user_df['time_since_last_action'].isna().sum():,}")
print(f"Zero values:      {(user_df['time_since_last_action'] == 0).sum():,}")
print(f"Negative values:  {(user_df['time_since_last_action'] < 0).sum():,}")

time_since_last_action
Missing values:   0
Zero values:      0
Negative values:  0


In [12]:
# ============================================================
# NUMERIC FEATURE SUMMARY
# ============================================================

numeric_features = [
    "candidate_count",
    "history_unique_query_count",
    "history_unique_wid_count",
    "candidate_click_rate",
    "candidate_cart_rate",
    "candidate_purchase_rate",
    "history_click_count",
    "history_cart_count",
    "history_order_count",
    "history_follow_count",
    "history_length",
    "candidate_seen_history_count",
    "time_since_last_action"
]


numeric_summary = pd.DataFrame({
    "Min": user_df[numeric_features].min(),
    "Max": user_df[numeric_features].max(),
    "Mean": user_df[numeric_features].mean(),
    "Median": user_df[numeric_features].median(),
    "SD": user_df[numeric_features].std(),
    "Missing": user_df[numeric_features].isna().sum()
})
numeric_summary = numeric_summary.round(5)

numeric_summary

,Min,Max,Mean,Median,SD,Missing
candidate_count,1.0,200.0,89.22466,90.00000,3.885736e+01,0
history_unique_query_count,1.0,1331.0,30.70042,18.00000,3.889628e+01,0
history_unique_wid_count,1.0,3647.0,99.64908,52.00000,1.398403e+02,0
candidate_click_rate,0.0,1.0,0.02060,0.01111,5.587000e-02,0
candidate_cart_rate,0.0,1.0,0.00803,0.00000,4.016000e-02,0
candidate_purchase_rate,0.0,1.0,0.00200,0.00000,1.781000e-02,0
history_click_count,0.0,3575.0,47.74116,17.00000,9.333230e+01,0
history_cart_count,0.0,3614.0,66.30868,29.00000,1.150793e+02,0
history_order_count,0.0,2962.0,39.21788,15.00000,7.040689e+01,0
history_follow_count,0.0,0.0,0.00000,0.00000,0.000000e+00,0


In [13]:
# ============================================================
# VALIDATION: Historical interaction counts
# ============================================================
# A. Candidate Label Distribution (Converted to int for cleaner display)
all_labels = [int(label) for sublist in user_df['candidate_label_list_parsed'] for label in sublist]
print("Candidate Label Dist (0:None, 1:Click, 2:Cart, 3:Purchase):")
print(pd.Series(all_labels).value_counts().sort_index())

# B. History Type Distribution
all_types = [t for sublist in user_df['history_type_list_parsed'] for t in sublist]
print("\nHistory Interaction Type Distribution:")
print(pd.Series(all_types).value_counts())


print("Historical CLICK")
print("Flattened count:",
      (pd.Series(all_types) == "CLICK").sum())
print("Sum of engineered counts:",
      user_df["history_click_count"].sum())

print("\nHistorical CART")
print("Flattened count:",
      (pd.Series(all_types) == "CART").sum())
print("Sum of engineered counts:",
      user_df["history_cart_count"].sum())

print("\nHistorical ORD")
print("Flattened count:",
      (pd.Series(all_types) == "ORD").sum())
print("Sum of engineered counts:",
      user_df["history_order_count"].sum())

print("\nHistorical FOLLOW")
print("Flattened count:",
      (pd.Series(all_types) == "FOLLOW").sum())
print("Sum of engineered counts:",
      user_df["history_follow_count"].sum())

# ============================================================
# VALIDATION: Candidate interaction counts
# ============================================================

labels = pd.Series(all_labels)

print("Candidate CLICK")
print("Flattened count:",
      (labels == 1).sum())
print("Recovered from engineered features:",
      (user_df["candidate_click_rate"] *
       user_df["candidate_count"]).sum())

print("\nCandidate CART")
print("Flattened count:",
      (labels == 2).sum())
print("Recovered from engineered features:",
      (user_df["candidate_cart_rate"] *
       user_df["candidate_count"]).sum())

print("\nCandidate PURCHASE")
print("Flattened count:",
      (labels == 3).sum())
print("Recovered from engineered features:",
      (user_df["candidate_purchase_rate"] *
       user_df["candidate_count"]).sum())

Candidate Label Dist (0:None, 1:Click, 2:Cart, 3:Purchase):
0    15189116
1      224487
2       75573
3       20836
Name: count, dtype: int64

History Interaction Type Distribution:
CART     11526504
CLICK     8298894
ORD       6817283
FLW         24579
Name: count, dtype: int64
Historical CLICK
Flattened count: 8298894
Sum of engineered counts: 8298894

Historical CART
Flattened count: 11526504
Sum of engineered counts: 11526504

Historical ORD
Flattened count: 6817283
Sum of engineered counts: 6817283

Historical FOLLOW
Flattened count: 0
Sum of engineered counts: 0
Candidate CLICK
Flattened count: 224487
Recovered from engineered features: 224487.0

Candidate CART
Flattened count: 75573
Recovered from engineered features: 75573.0

Candidate PURCHASE
Flattened count: 20836
Recovered from engineered features: 20836.0


## Categorical Features

In [14]:
# ============================================================
# CATEGORICAL / ID FEATURE SUMMARY
# ============================================================

categorical_features = [
    "candidate_wid_list_parsed",
    "candidate_label_list_parsed",
    "history_qry_list_parsed",
    "history_wid_list_parsed",
    "history_type_list_parsed"
]

categorical_summary = []

for col in categorical_features:

    # Missing lists at row level
    missing_count = user_df[col].isna().sum()

    # Flatten all lists
    values = user_df[col].dropna().explode()

    # Remove missing values within lists
    values = values.dropna()

    # Distinct categories
    distinct_count = values.nunique()

    # Representative values
    representative_values = ", ".join(
        map(str, values.value_counts().head(5).index)
    )

    categorical_summary.append({
        "Feature": col,
        "Distinct Categories": distinct_count,
        "Representative Values": representative_values,
        "Missing": missing_count
    })

# ============================================================
# CANDIDATE LABEL DISTRIBUTION
# 0 = None, 1 = Click, 2 = Cart, 3 = Purchase
# ============================================================

all_labels = [
    int(label)
    for sublist in user_df["candidate_label_list_parsed"]
    for label in sublist
]

candidate_label_dist = pd.Series(all_labels).value_counts().sort_index()

candidate_label_summary = pd.DataFrame({
    "Count": candidate_label_dist,
    "Percent": candidate_label_dist / candidate_label_dist.sum() * 100
})

candidate_label_summary["Percent"] = candidate_label_summary["Percent"].round(5)

print("Candidate Label Distribution:")
display(candidate_label_summary)

# ============================================================
# HISTORY INTERACTION TYPE DISTRIBUTION
# ============================================================

all_types = [
    interaction
    for sublist in user_df["history_type_list_parsed"]
    for interaction in sublist
]

history_type_dist = pd.Series(all_types).value_counts()

history_type_summary = pd.DataFrame({
    "Count": history_type_dist,
    "Percent": history_type_dist / history_type_dist.sum() * 100
})

history_type_summary["Percent"] = history_type_summary["Percent"].round(5)

print("History Interaction Type Distribution:")
display(history_type_summary)


categorical_summary = pd.DataFrame(categorical_summary)

categorical_summary

Candidate Label Distribution:


,Count,Percent
0,15189116,97.93104
1,224487,1.44737
2,75573,0.48725
3,20836,0.13434


History Interaction Type Distribution:


,Count,Percent
CART,11526504,43.22343
CLICK,8298894,31.12016
ORD,6817283,25.56424
FLW,24579,0.09217


,Feature,Distinct Categories,Representative Values,Missing
0,candidate_wid_list_parsed,8362535,"26194387.0, 64338053.0, 71524113.0, 99727072.0...",0
1,candidate_label_list_parsed,4,"0.0, 1.0, 2.0, 3.0",0
2,history_qry_list_parsed,1691283,"-1, 61485993, 59054959, 88843206, 94032448",0
3,history_wid_list_parsed,5584543,"97745171, 99727072, 81138348, 24157445, 76456085",0
4,history_type_list_parsed,4,"CART, CLICK, ORD, FLW",0


## Product Metadata: Chunked Loading & Descriptive Statistics

product_meta_data.txt is ~4.8GB (12.14M rows). Loading it with a single `pd.read_csv` risks exhausting RAM (a `dtype=str` load can balloon to 3-5x the raw file size). Instead we stream it in chunks with `pd.read_csv(..., chunksize=...)` and maintain O(1)-memory running statistics per column (Welford's online algorithm for numeric/length stats, sets/counters for categorical distinct counts and top values) rather than holding the full DataFrame in memory.

In [1]:
# ============================================================
# Chunked ingestion + streaming descriptive statistics for product_meta_data.txt
# Avoids loading the full 4.8GB / 12.14M-row file into memory at once.
# ============================================================
import time
from collections import Counter

# name/brand_name/cate_name_* fields are sequences of anonymized term ids
# concatenated with the octal \030 separator (= chr(24) = \x18), per the
# JDsearch README. candidate/history list fields (in user_df) use '_' instead;
# this \x18 separator is specific to the term-id fields in product_meta_data.txt.
TERM_SEP = "\x18"

PRODUCT_COLS = [
    "wid", "name", "brand_id", "brand_name",
    "cate_id_1", "cate_name_1", "cate_id_2", "cate_name_2",
    "cate_id_3", "cate_name_3", "cate_id_4", "cate_name_4",
    "shop_id",
]

# wid is (by design) a unique product id, so an exact distinct-count set for it
# would cost ~1-2GB for little insight; we verify its uniqueness via row_count
# instead and skip it here. shop_id/brand_id/cate_id_* have far lower
# cardinality and are cheap to track exactly.
ID_COLS = ["brand_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4", "shop_id"]
TEXT_COLS = ["name", "brand_name", "cate_name_1", "cate_name_2", "cate_name_3", "cate_name_4"]

CHUNK_SIZE = 200_000


class OnlineStats:
    """Welford's algorithm: running mean/std/min/max in O(1) memory (no list of values kept)."""
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0
        self.min = None
        self.max = None

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2
        self.min = x if self.min is None else min(self.min, x)
        self.max = x if self.max is None else max(self.max, x)

    @property
    def std(self):
        return (self.M2 / self.n) ** 0.5 if self.n > 1 else 0.0


def parse_term_list(val):
    if pd.isna(val) or val == "":
        return []
    return val.split(TERM_SEP)


row_count = 0
missing_counts = {c: 0 for c in PRODUCT_COLS}
distinct_ids = {c: set() for c in ID_COLS}
id_top_values = {c: Counter() for c in ID_COLS}
text_len_stats = {c: OnlineStats() for c in TEXT_COLS}
text_vocab = {c: set() for c in TEXT_COLS}

reader = pd.read_csv(
    PRODUCT_TXT, sep="\t", names=PRODUCT_COLS, header=0,
    dtype=str, chunksize=CHUNK_SIZE, low_memory=False,
)

start = time.time()
for i, chunk in enumerate(reader):
    row_count += len(chunk)

    for c in PRODUCT_COLS:
        missing_counts[c] += int(chunk[c].isna().sum())

    for c in ID_COLS:
        distinct_ids[c].update(chunk[c].dropna().unique())
        id_top_values[c].update(chunk[c].dropna())

    for c in TEXT_COLS:
        for terms in chunk[c].dropna().apply(parse_term_list):
            text_len_stats[c].update(len(terms))
            text_vocab[c].update(terms)

    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"chunk {i+1}: {row_count:,} rows, {elapsed:.1f}s elapsed, {row_count/elapsed:,.0f} rows/s")

elapsed = time.time() - start
print(f"\nDONE: {row_count:,} rows processed in {elapsed:.1f}s ({row_count/elapsed:,.0f} rows/s)")
print(f"wid: {row_count:,} rows (treated as primary key, exact distinct-count set skipped to save memory)")

chunk 10: 2,000,000 rows, 17.1s elapsed, 117,244 rows/s
chunk 20: 4,000,000 rows, 35.1s elapsed, 113,927 rows/s
chunk 30: 6,000,000 rows, 55.0s elapsed, 109,013 rows/s
chunk 40: 8,000,000 rows, 75.0s elapsed, 106,628 rows/s
chunk 50: 10,000,000 rows, 95.6s elapsed, 104,635 rows/s
chunk 60: 12,000,000 rows, 117.1s elapsed, 102,451 rows/s

DONE: 12,141,247 rows processed in 118.5s (102,458 rows/s)
wid: 12,141,247 rows (treated as primary key, exact distinct-count set skipped to save memory)


In [2]:
# ============================================================
# Categorical (ID) feature summary for product metadata
# ============================================================
id_summary = pd.DataFrame({
    "Distinct Categories": {c: len(distinct_ids[c]) for c in ID_COLS},
    "Missing": {c: missing_counts[c] for c in ID_COLS},
    "Representative Values": {
        c: ", ".join(v for v, _ in id_top_values[c].most_common(5))
        for c in ID_COLS
    },
})

print(f"wid (product id): {row_count:,} distinct values (1 per row, primary key)")
id_summary

wid (product id): 12,141,247 distinct values (1 per row, primary key)


,Distinct Categories,Missing,Representative Values
brand_id,182585,0,"38645246, 54289642, 33241674, 67072585, 98209701"
cate_id_1,76,0,"68949654, 24093831, 920973, 98371044, 31606173"
cate_id_2,671,0,"92351230, 18907346, 33276684, 71940523, 38845680"
cate_id_3,5819,0,"72526272, 10287851, 64184092, 21386661, 94549943"
cate_id_4,7137,0,"32130995, 462318, 23794926, 97122609, 76679610"
shop_id,250935,0,"-1, 21749031, 54486560, 58139116, 74969976"


In [3]:
# ============================================================
# Text / corpus feature summary: term-id sequence length distributions
# and vocabulary sizes for name / brand_name / category name fields
# ============================================================
text_summary = pd.DataFrame({
    "Mean Length": {c: round(text_len_stats[c].mean, 3) for c in TEXT_COLS},
    "Std Length": {c: round(text_len_stats[c].std, 3) for c in TEXT_COLS},
    "Min Length": {c: text_len_stats[c].min for c in TEXT_COLS},
    "Max Length": {c: text_len_stats[c].max for c in TEXT_COLS},
    "Vocab Size (distinct term ids)": {c: len(text_vocab[c]) for c in TEXT_COLS},
    "Missing": {c: missing_counts[c] for c in TEXT_COLS},
})
text_summary

,Mean Length,Std Length,Min Length,Max Length,Vocab Size (distinct term ids),Missing
name,27.913,8.056,1,78,986404,0
brand_name,2.561,1.731,1,31,192684,941660
cate_name_1,1.774,0.663,1,9,128,0
cate_name_2,1.917,0.690,1,7,751,0
cate_name_3,1.913,0.938,1,15,4996,0
cate_name_4,1.945,0.951,1,15,5926,0
